# Stellar Coordinate Explorer - Data Loading and Inspection (Unbiased)

## Objective
Load the randomly selected Gaia DR3 catalogue and perform an initial inspection.

## Why this matters
Random sampling avoids the biases of simply taking the first N rows from a database query. This dataset will serve as the primary, scientifically robust sample for the rest of the project.

## Dataset
- Source: Gaia DR3
- Subset: ~ 10,000 stars
- Filters:
  - $Magnitude\ \lt 10$
  - $Parallax\ \gt5\ mas\ (\approx 200\ parsecs)$

The Dataset was obtained from the [Gaia archive](https://gea.esac.esa.int/archive/) using the following SQL query:
```SQL
SELECT TOP 10000
    source_id,
    ra, 
    dec, 
    parallax,
    phot_g_mean_mag,
    bp_rp
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 10
AND parallax > 5
ORDER BY random_index
```
The file with this data is named `gaia_subset_random.fits`

## Goals for Today
- Load dataset
- Inspect structure
- Understand key columns
- Print first 5 rows
- Create a filtered table with values below a magnitude
- Print the 5 brightest stars with their magnitudes
- Print the mean and median of the full sample
- Save the filtered table for Day 4

## Checkpoint 
- Table loaded without errors
- Table show 10,000 rows with correct dtypes
- First 5 rows printed
- Mean and median magnitude computed

## Code

### Loading and Inspection

In [1]:
from astropy.table import Table
import numpy as np

In [2]:
# Load the random FITS table
table = Table.read("../../data/gaia_subset_random.fits")

# Get summary information
print(f"{'#' * 20} Summary information {'#' * 20}")
table.info()

#################### Summary information ####################
<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    12


12 of the randomly selected sources appear to have missing values in the `bp_rp` column. `bp_rp` is difference between the integrated magintude of the source measured in the blue photometer and red photometer.

### Key Columns

Looking at the dataset's columns closely:

In [3]:
print(f"\n{'#' * 20} Show table column names {'#' * 20}")
print(table.columns) 
print(table.colnames)


#################### Show table column names ####################
<TableColumns names=('source_id','ra','dec','parallax','phot_g_mean_mag','bp_rp')>
['source_id', 'ra', 'dec', 'parallax', 'phot_g_mean_mag', 'bp_rp']


We have the `source_id` used to identify the star, `ra` and `dec` which is the right ascension and declination coordinates of the source (using the ICRS coordinate frame), `parallax` (for determining distance to source), `phot_g_mean_mag` (mean G-band magnitude) and `bp_rp`. Therefore key columns can be summarized as:

- `ra`, `dec`: Equitorial coordinates (degrees)
- `parallax`: Used to estimate distance
- `phot_g_mean_mag`: Used to determine brightness
- `bp_rp`: Color index useful for indicating temperature.


Now, looking at the first 5 rows:

In [4]:
print(f"{'#' * 20} First five rows {'#' * 20}")
table[:5]

#################### First five rows ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
2916442949023646464,89.08891529461751,-23.375564356338113,6.174716638660832,9.528021,0.71763134
486087738386602624,89.75122195274709,71.33384612083127,6.786864259555966,8.923034,0.5152149
86268270726618624,37.93089459029714,19.29079672106284,5.567776813866466,8.103913,1.2267585
6245995261529926784,245.12519193816152,-18.658756095050595,12.422733521538417,9.504698,1.0528879
4599710073355218048,259.3530395685651,30.4113148729394,7.205622033945202,9.934753,0.8441076


The `ra` and `dec` are in degrees, `parallax` is in milliarcsecs (mas) and both `phot_g_mean_mag` and `bp_rp` are magnitudes. 

### Brightest Stars

Let's filter to obtain the stars with magnitudes `phot_g_mean_mag` $\lt 6$

In [7]:
mask = table["phot_g_mean_mag"] < 6
print(f"{'#' * 20} Stars with G-band mean magnitudes less than 6 {'#' * 20}")
bright_table = table[mask]
bright_table

#################### Stars with G-band mean magnitudes less than 6 ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
3380338312213644416,102.99975006953139,23.601682615516822,7.274806414892579,5.1445856,1.6490936
5696837272733204352,126.22978552915332,-23.153614962557757,5.170321129081684,5.6464987,0.12439251
3208587277330249856,78.20063354346843,-6.057326650343134,8.015840787260867,5.6570015,1.1035862
2913650636166401280,91.63368033863665,-23.11096284344187,6.595676739416982,5.4475493,0.11610079
5358392578511815168,154.1669357333221,-51.20458615375942,5.1126735435775545,4.7541423,3.0313478
2246581329639103872,304.3858330649528,66.85501172532437,57.20405217305612,5.7695527,0.77145576
816649002967779584,135.1573578568694,41.78175608312542,53.36175020467222,3.959608,0.6138511
1193030490492925824,239.11469743534354,15.655912590196278,89.56466413447797,3.694426,0.7034795


Let's print the 5 brightest stars

In [8]:
sorted_table = bright_table.copy()
sorted_table.sort('phot_g_mean_mag')
print(f"{'#' * 20} Five brightest stars {'#' * 20}")
sorted_table[:5]

#################### Five brightest stars ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
1279798794197267072,221.24648617076684,27.074315757773466,13.82667260913517,2.1833525,1.1840065
6227443304915069056,226.0172005141403,-25.2821590562471,12.538654525627177,2.3034124,1.9100549
4337352305320149760,249.28979755823173,-10.566976077121021,7.408845727983025,2.5565956,0.44186306
2962546605447869184,76.36538394458108,-22.371367602513757,15.599908298075055,2.6534927,1.5497768
779106556394253824,167.41547770774767,44.4983642995888,23.227226550456237,2.68003,1.3199959


In [9]:
## Save the brightest stars
sorted_table.write("../../data/bright_stars_filtered_random.fits")

### Light Statistics (Mean and Median)

Now we obtain the mean and median G-band magnitudes of the whole set:

In [12]:
mean_mag = np.mean(table['phot_g_mean_mag'])
median_mag = np.median(table['phot_g_mean_mag'])
max_mag = np.max(table['phot_g_mean_mag'])
min_mag = np.min(table['phot_g_mean_mag'])
std_mag = np.std(table['phot_g_mean_mag'])

print(f"{'#' * 20} Mean and Median G-band mean magnitudes of the 10000 sources {'#' * 20}")
print(f"Mean G magnitude: {mean_mag}")
print(f"Median G magnitude: {median_mag}")

print(f"{'#' * 20} Minimum and maximum G-band mean magnitudes of the 10000 sources {'#' * 20}")
print(f"Maximum G magnitude: {max_mag}")
print(f"Minimum G magnitude: {min_mag}")

print(f"{'#' * 20} Standard deviation of G-band mean magnitudes of the 10000 sources {'#' * 20}")
print(f"Standard deviation of G magnitude: {std_mag}")

#################### Mean and Median G-band mean magnitudes of the 10000 sources ####################
Mean G magnitude: 8.643251419067383
Median G magnitude: 8.992260932922363
#################### Minimum and maximum G-band mean magnitudes of the 10000 sources ####################
Maximum G magnitude: 9.99980640411377
Minimum G magnitude: 2.183352470397949
#################### Standard deviation of G-band mean magnitudes of the 10000 sources ####################
Standard deviation of G magnitude: 1.2222732305526733


## Basic Observations

The magnitude of the stars appear to span a wide range from a mean G magnitude of 2 to approximately 10 with most sources having a faint magnitude given the mean G-band magnitude of 8.6 and a standard deviation of 1.2.